## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from sem_rag.config.settings import *

In [ ]:
import json
import pandas as pd

from haystack import Pipeline, component
from haystack.utils import Secret
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

# Queries

## Quality Control of Metadata

### Numeric Fields

#### Pixel Size

In [ ]:
PIXEL_SIZE_CASES = [
    {
        "id": "PIX-01",
        "category": "numeric_approx",
        "query": "Show images with a pixel size of approximately 100 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "PIX-02",
        "category": "numeric_approx",
        "query": "Find images with a pixel size of about 50 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 45.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 55.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    {
        "id": "PIX-03",
        "category": "numeric_comparison",
        "query": "Show images with a pixel size greater than or equal to 100 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.pixel_size_nm",
                        "operator": ">=",
                        "value": 100.0,
                    }
                ],
            }
        },
    },
    {
        "id": "PIX-04",
        "category": "numeric_comparison",
        "query": "Find images with a pixel size of at least 200 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.pixel_size_nm",
                        "operator": ">=",
                        "value": 200.0,
                    }
                ],
            }
        },
    },

    {
        "id": "PIX-05",
        "category": "numeric_comparison",
        "query": "Show images with a pixel size less than or equal to 100 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.pixel_size_nm",
                        "operator": "<=",
                        "value": 100.0,
                    }
                ],
            }
        },
    },
    {
        "id": "PIX-06",
        "category": "numeric_comparison",
        "query": "Find images with a pixel size of at most 25 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.pixel_size_nm",
                        "operator": "<=",
                        "value": 25.0,
                    }
                ],
            }
        },
    },

    {
        "id": "PIX-07",
        "category": "numeric_between",
        "query": "Show images with a pixel size between 50 nm and 100 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 50.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 100.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "PIX-08",
        "category": "numeric_between",
        "query": "Find images with a pixel size between 200 nm and 300 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 200.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 300.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Magnification

In [ ]:
MAGNIFICATION_CASES = [
    {
        "id": "MAG-01",
        "category": "numeric_approx",
        "query": "Show images with a magnification of approximately 500x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.magnification_x",
                                "operator": ">=",
                                "value": 450.0,
                            },
                            {
                                "field": "meta.magnification_x",
                                "operator": "<=",
                                "value": 550.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "MAG-02",
        "category": "numeric_approx",
        "query": "Find images with a magnification of 1000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.magnification_x",
                                "operator": ">=",
                                "value": 900.0,
                            },
                            {
                                "field": "meta.magnification_x",
                                "operator": "<=",
                                "value": 1100.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    {
        "id": "MAG-03",
        "category": "numeric_comparison",
        "query": "Show images with a magnification greater than or equal to 5000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 5000.0,
                    }
                ],
            }
        },
    },
    {
        "id": "MAG-04",
        "category": "numeric_comparison",
        "query": "Find images with a magnification of at least 10000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 10000.0,
                    }
                ],
            }
        },
    },

    {
        "id": "MAG-05",
        "category": "numeric_comparison",
        "query": "Show images with a magnification less than or equal to 1000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.magnification_x",
                        "operator": "<=",
                        "value": 1000.0,
                    }
                ],
            }
        },
    },
    {
        "id": "MAG-06",
        "category": "numeric_comparison",
        "query": "Find images with a magnification of at most 5000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.magnification_x",
                        "operator": "<=",
                        "value": 5000.0,
                    }
                ],
            }
        },
    },

    {
        "id": "MAG-07",
        "category": "between",
        "query": "Show images with a magnification between 500x and 1000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.magnification_x",
                                "operator": ">=",
                                "value": 500.0,
                            },
                            {
                                "field": "meta.magnification_x",
                                "operator": "<=",
                                "value": 1000.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "MAG-08",
        "category": "between",
        "query": "Find images with a magnification between 5000x and 10000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.magnification_x",
                                "operator": ">=",
                                "value": 5000.0,
                            },
                            {
                                "field": "meta.magnification_x",
                                "operator": "<=",
                                "value": 10000.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Accelerating Voltage

In [ ]:
ACCELERATING_VOLTAGE_CASES = [
    {
        "id": "AV-01",
        "category": "numeric_approx",
        "query": "Show images with an accelerating voltage of approximately 10 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "AV-02",
        "category": "numeric_approx",
        "query": "Find images with an accelerating voltage of 5 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 4.5,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 5.5,
                            },
                        ],
                    }
                ],
            }
        },
    },

    {
        "id": "AV-03",
        "category": "numeric_comparison",
        "query": "Show images with an accelerating voltage greater than or equal to 15 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.accelerating_voltage_kv",
                        "operator": ">=",
                        "value": 15.0,
                    }
                ],
            }
        },
    },
    {
        "id": "AV-04",
        "category": "numeric_comparison",
        "query": "Find images with an accelerating voltage of at least 20 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.accelerating_voltage_kv",
                        "operator": ">=",
                        "value": 20.0,
                    }
                ],
            }
        },
    },

    {
        "id": "AV-05",
        "category": "numeric_comparison",
        "query": "Show images with an accelerating voltage less than or equal to 5 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.accelerating_voltage_kv",
                        "operator": "<=",
                        "value": 5.0,
                    }
                ],
            }
        },
    },
    {
        "id": "AV-06",
        "category": "numeric_comparison",
        "query": "Find images with an accelerating voltage of at most 10 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.accelerating_voltage_kv",
                        "operator": "<=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },

    {
        "id": "AV-07",
        "category": "between",
        "query": "Show images with an accelerating voltage between 5 kV and 10 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "AV-08",
        "category": "between",
        "query": "Find images with an accelerating voltage between 10 kV and 20 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 10.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 20.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Working Distance

In [ ]:
WORKING_DISTANCE_CASES = [
    {
        "id": "WD-01",
        "category": "numeric_approx",
        "query": "Show images with a working distance of approximately 10 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.working_distance_mm",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.working_distance_mm",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "WD-02",
        "category": "numeric_approx",
        "query": "Find images with a working distance of 5 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.working_distance_mm",
                                "operator": ">=",
                                "value": 4.5,
                            },
                            {
                                "field": "meta.working_distance_mm",
                                "operator": "<=",
                                "value": 5.5,
                            },
                        ],
                    }
                ],
            }
        },
    },

    {
        "id": "WD-03",
        "category": "numeric_comparison",
        "query": "Show images with a working distance greater than or equal to 10 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.working_distance_mm",
                        "operator": ">=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },
    {
        "id": "WD-04",
        "category": "numeric_comparison",
        "query": "Find images with a working distance of at least 15 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.working_distance_mm",
                        "operator": ">=",
                        "value": 15.0,
                    }
                ],
            }
        },
    },

    {
        "id": "WD-05",
        "category": "numeric_comparison",
        "query": "Show images with a working distance less than or equal to 5 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.working_distance_mm",
                        "operator": "<=",
                        "value": 5.0,
                    }
                ],
            }
        },
    },
    {
        "id": "WD-06",
        "category": "numeric_comparison",
        "query": "Find images with a working distance of at most 10 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.working_distance_mm",
                        "operator": "<=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },

    {
        "id": "WD-07",
        "category": "between",
        "query": "Show images with a working distance between 5 mm and 10 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.working_distance_mm",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.working_distance_mm",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "WD-08",
        "category": "between",
        "query": "Find images with a working distance between 10 mm and 20 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.working_distance_mm",
                                "operator": ">=",
                                "value": 10.0,
                            },
                            {
                                "field": "meta.working_distance_mm",
                                "operator": "<=",
                                "value": 20.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Brightness

In [ ]:
BRIGHTNESS_CASES = [
    {
        "id": "BRI-01",
        "category": "numeric_approx",
        "query": "Show images with a brightness of approximately 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.brightness_percent",
                                "operator": ">=",
                                "value": 45.0,
                            },
                            {
                                "field": "meta.brightness_percent",
                                "operator": "<=",
                                "value": 55.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "BRI-02",
        "category": "numeric_approx",
        "query": "Find images with a brightness of 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.brightness_percent",
                                "operator": ">=",
                                "value": 67.5,
                            },
                            {
                                "field": "meta.brightness_percent",
                                "operator": "<=",
                                "value": 82.5,
                            },
                        ],
                    }
                ],
            }
        },
    },

    {
        "id": "BRI-03",
        "category": "numeric_comparison",
        "query": "Show images with a brightness greater than or equal to 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.brightness_percent",
                        "operator": ">=",
                        "value": 50.0,
                    }
                ],
            }
        },
    },
    {
        "id": "BRI-04",
        "category": "numeric_comparison",
        "query": "Find images with a brightness of at least 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.brightness_percent",
                        "operator": ">=",
                        "value": 75.0,
                    }
                ],
            }
        },
    },

    {
        "id": "BRI-05",
        "category": "numeric_comparison",
        "query": "Show images with a brightness less than or equal to 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.brightness_percent",
                        "operator": "<=",
                        "value": 50.0,
                    }
                ],
            }
        },
    },
    {
        "id": "BRI-06",
        "category": "numeric_comparison",
        "query": "Find images with a brightness of at most 25%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.brightness_percent",
                        "operator": "<=",
                        "value": 25.0,
                    }
                ],
            }
        },
    },

    {
        "id": "BRI-07",
        "category": "between",
        "query": "Show images with a brightness between 25% and 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.brightness_percent",
                                "operator": ">=",
                                "value": 25.0,
                            },
                            {
                                "field": "meta.brightness_percent",
                                "operator": "<=",
                                "value": 50.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "BRI-08",
        "category": "between",
        "query": "Find images with a brightness between 50% and 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.brightness_percent",
                                "operator": ">=",
                                "value": 50.0,
                            },
                            {
                                "field": "meta.brightness_percent",
                                "operator": "<=",
                                "value": 75.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Contrast

In [ ]:
CONTRAST_CASES = [
    {
        "id": "CON-01",
        "category": "numeric_approx",
        "query": "Show images with a contrast of approximately 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.contrast_percent",
                                "operator": ">=",
                                "value": 45.0,
                            },
                            {
                                "field": "meta.contrast_percent",
                                "operator": "<=",
                                "value": 55.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "CON-02",
        "category": "numeric_approx",
        "query": "Find images with a contrast of 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.contrast_percent",
                                "operator": ">=",
                                "value": 67.5,
                            },
                            {
                                "field": "meta.contrast_percent",
                                "operator": "<=",
                                "value": 82.5,
                            },
                        ],
                    }
                ],
            }
        },
    },

    {
        "id": "CON-03",
        "category": "numeric_comparison",
        "query": "Show images with a contrast greater than or equal to 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.contrast_percent",
                        "operator": ">=",
                        "value": 50.0,
                    }
                ],
            }
        },
    },
    {
        "id": "CON-04",
        "category": "numeric_comparison",
        "query": "Find images with a contrast of at least 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.contrast_percent",
                        "operator": ">=",
                        "value": 75.0,
                    }
                ],
            }
        },
    },

    {
        "id": "CON-05",
        "category": "numeric_comparison",
        "query": "Show images with a contrast less than or equal to 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.contrast_percent",
                        "operator": "<=",
                        "value": 50.0,
                    }
                ],
            }
        },
    },
    {
        "id": "CON-06",
        "category": "numeric_comparison",
        "query": "Find images with a contrast of at most 25%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.contrast_percent",
                        "operator": "<=",
                        "value": 25.0,
                    }
                ],
            }
        },
    },

    {
        "id": "CON-07",
        "category": "between",
        "query": "Show images with a contrast between 25% and 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.contrast_percent",
                                "operator": ">=",
                                "value": 25.0,
                            },
                            {
                                "field": "meta.contrast_percent",
                                "operator": "<=",
                                "value": 50.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "CON-08",
        "category": "between",
        "query": "Find images with a contrast between 50% and 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.contrast_percent",
                                "operator": ">=",
                                "value": 50.0,
                            },
                            {
                                "field": "meta.contrast_percent",
                                "operator": "<=",
                                "value": 75.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Scan Speed

In [ ]:
SCAN_SPEED_CASES = [
    {
        "id": "SS-01",
        "category": "numeric_approx",
        "query": "Show images with a scan speed setting of approximately 5.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": ">=",
                                "value": 4.5,
                            },
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": "<=",
                                "value": 5.5,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "SS-02",
        "category": "numeric_approx",
        "query": "Find images with a scan speed setting of 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    {
        "id": "SS-03",
        "category": "numeric_comparison",
        "query": "Show images with a scan speed setting greater than or equal to 5.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.scan_speed_setting",
                        "operator": ">=",
                        "value": 5.0,
                    }
                ],
            }
        },
    },
    {
        "id": "SS-04",
        "category": "numeric_comparison",
        "query": "Find images with a scan speed setting of at least 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.scan_speed_setting",
                        "operator": ">=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },

    {
        "id": "SS-05",
        "category": "numeric_comparison",
        "query": "Show images with a scan speed setting less than or equal to 5.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.scan_speed_setting",
                        "operator": "<=",
                        "value": 5.0,
                    }
                ],
            }
        },
    },
    {
        "id": "SS-06",
        "category": "numeric_comparison",
        "query": "Find images with a scan speed setting of at most 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.scan_speed_setting",
                        "operator": "<=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },

    {
        "id": "SS-07",
        "category": "between",
        "query": "Show images with a scan speed setting between 5 and 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "SS-08",
        "category": "between",
        "query": "Find images with a scan speed setting between 10 and 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": ">=",
                                "value": 10.0,
                            },
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": "<=",
                                "value": 20.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Frame Average Count

In [ ]:
FRAME_AVERAGE_COUNT_CASES = [
    {
        "id": "FA-01",
        "category": "numeric_approx",
        "query": "Show images with a frame average count of approximately 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.frame_average_count",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.frame_average_count",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "FA-02",
        "category": "numeric_approx",
        "query": "Find images with a frame average count of 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.frame_average_count",
                                "operator": ">=",
                                "value": 18.0,
                            },
                            {
                                "field": "meta.frame_average_count",
                                "operator": "<=",
                                "value": 22.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "FA-03",
        "category": "numeric_comparison",
        "query": "Show images with a frame average count greater than or equal to 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.frame_average_count",
                        "operator": ">=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },
    {
        "id": "FA-04",
        "category": "numeric_comparison",
        "query": "Find images with a frame average count of at least 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.frame_average_count",
                        "operator": ">=",
                        "value": 20.0,
                    }
                ],
            }
        },
    },
    {
        "id": "FA-05",
        "category": "numeric_comparison",
        "query": "Show images with a frame average count less than or equal to 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.frame_average_count",
                        "operator": "<=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },
    {
        "id": "FA-06",
        "category": "numeric_comparison",
        "query": "Find images with a frame average count of at most 5.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.frame_average_count",
                        "operator": "<=",
                        "value": 5.0,
                    }
                ],
            }
        },
    },
    {
        "id": "FA-07",
        "category": "between",
        "query": "Show images with a frame average count between 5 and 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.frame_average_count",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.frame_average_count",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "FA-08",
        "category": "between",
        "query": "Find images with a frame average count between 10 and 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.frame_average_count",
                                "operator": ">=",
                                "value": 10.0,
                            },
                            {
                                "field": "meta.frame_average_count",
                                "operator": "<=",
                                "value": 20.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

#### Line Average Count

In [ ]:
LINE_AVERAGE_COUNT_CASES = [
    {
        "id": "LA-01",
        "category": "numeric_approx",
        "query": "Show images with a line average count of approximately 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.line_average_count",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.line_average_count",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "LA-02",
        "category": "numeric_approx",
        "query": "Find images with a line average count of 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.line_average_count",
                                "operator": ">=",
                                "value": 18.0,
                            },
                            {
                                "field": "meta.line_average_count",
                                "operator": "<=",
                                "value": 22.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "LA-03",
        "category": "numeric_comparison",
        "query": "Show images with a line average count greater than or equal to 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.line_average_count",
                        "operator": ">=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },
    {
        "id": "LA-04",
        "category": "numeric_comparison",
        "query": "Find images with a line average count of at least 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.line_average_count",
                        "operator": ">=",
                        "value": 20.0,
                    }
                ],
            }
        },
    },
    {
        "id": "LA-05",
        "category": "numeric_comparison",
        "query": "Show images with a line average count less than or equal to 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.line_average_count",
                        "operator": "<=",
                        "value": 10.0,
                    }
                ],
            }
        },
    },
    {
        "id": "LA-06",
        "category": "numeric_comparison",
        "query": "Find images with a line average count of at most 5.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.line_average_count",
                        "operator": "<=",
                        "value": 5.0,
                    }
                ],
            }
        },
    },
    {
        "id": "LA-07",
        "category": "between",
        "query": "Show images with a line average count between 5 and 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.line_average_count",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.line_average_count",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "LA-08",
        "category": "between",
        "query": "Find images with a line average count between 10 and 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.line_average_count",
                                "operator": ">=",
                                "value": 10.0,
                            },
                            {
                                "field": "meta.line_average_count",
                                "operator": "<=",
                                "value": 20.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

### Categorical Fields

#### Category

In [ ]:
CATEGORY_CASES = [
    {
        "id": "CAT-01",
        "category": "categorical",
        "query": "Show images from the Porous_Sponge category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-02",
        "category": "categorical",
        "query": "Find images in the Porous_Sponge category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-03",
        "category": "categorical",
        "query": "Show images from the Patterned_surface category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Patterned_surface",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-04",
        "category": "categorical",
        "query": "Find images in the Patterned_surface category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Patterned_surface",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-05",
        "category": "categorical",
        "query": "Show images from the Particles category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Particles",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-06",
        "category": "categorical",
        "query": "Find images in the Particles category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Particles",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-07",
        "category": "categorical",
        "query": "Show images from the Films_Coated_Surface category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Films_Coated_Surface",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-08",
        "category": "categorical",
        "query": "Find images in the Films_Coated_Surface category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Films_Coated_Surface",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-09",
        "category": "categorical",
        "query": "Show images from the Powder category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Powder",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-10",
        "category": "categorical",
        "query": "Find images in the Powder category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Powder",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-11",
        "category": "categorical",
        "query": "Show images from the Tips category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Tips",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-12",
        "category": "categorical",
        "query": "Find images in the Tips category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Tips",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-13",
        "category": "categorical",
        "query": "Show images from the Nanowires category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-14",
        "category": "categorical",
        "query": "Find images in the Nanowires category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-15",
        "category": "categorical",
        "query": "Show images from the Biological category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Biological",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-16",
        "category": "categorical",
        "query": "Find images in the Biological category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Biological",
                    }
                ],
            }
        },
    },

    {
        "id": "CAT-17",
        "category": "categorical",
        "query": "Show images from the MEMS_devices_and_electrodes category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "MEMS_devices_and_electrodes",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-18",
        "category": "categorical",
        "query": "Find images in the MEMS_devices_and_electrodes category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "MEMS_devices_and_electrodes",
                    }
                ],
            }
        },
    },
    
    {
        "id": "CAT-19",
        "category": "categorical",
        "query": "Show images from the Fibres category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Fibres",
                    }
                ],
            }
        },
    },
    {
        "id": "CAT-20",
        "category": "categorical",
        "query": "Find images in the Fibres category.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Fibres",
                    }
                ],
            }
        },
    },
]

#### Detector

In [ ]:
DETECTOR_CASES = [
    {
        "id": "DET-01",
        "category": "categorical",
        "query": "Show images taken with the InLens detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    }
                ],
            }
        },
    },
    {
        "id": "DET-02",
        "category": "categorical",
        "query": "Find images using the InLens detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    }
                ],
            }
        },
    },

    {
        "id": "DET-03",
        "category": "categorical",
        "query": "Show images taken with the SE2 detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "SE2",
                    }
                ],
            }
        },
    },
    {
        "id": "DET-04",
        "category": "categorical",
        "query": "Find images using the SE2 detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "SE2",
                    }
                ],
            }
        },
    },
    
    {
        "id": "DET-05",
        "category": "categorical",
        "query": "Show images taken with the HDAsB detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "HDAsB",
                    }
                ],
            }
        },
    },
    {
        "id": "DET-06",
        "category": "categorical",
        "query": "Find images using the HDAsB detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "HDAsB",
                    }
                ],
            }
        },
    },
]

### Multiple Metadata Constraints

#### Numeric + Numeric

In [ ]:
NUMERIC_NUMERIC_CASES = [
    {
        "id": "NN-01",
        "category": "multiple_numeric",
        "query": "Show images with a pixel size of approximately 100 nm and a magnification of at least 500x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    },
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 500.0,
                    },
                ],
            }
        },
    },
    {
        "id": "NN-02",
        "category": "multiple_numeric",
        "query": "Find images with an accelerating voltage between 5 kV and 10 kV and a working distance of at most 15 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    },
                    {
                        "field": "meta.working_distance_mm",
                        "operator": "<=",
                        "value": 15.0,
                    },
                ],
            }
        },
    },
]

#### Categorical + Categorical

In [ ]:
CATEGORICAL_CATEGORICAL_CASES = [
    {
        "id": "CC-01",
        "category": "multiple_categorical",
        "query": "Show images from the Porous_Sponge category taken with the InLens detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    },
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    },
                ],
            }
        },
    },
    {
        "id": "CC-02",
        "category": "multiple_categorical",
        "query": "Find images from the Nanowires category using the SE2 detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    },
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "SE2",
                    },
                ],
            }
        },
    },
]

#### Numeric + Categorical

In [ ]:
NUMERIC_CATEGORICAL_CASES = [
    {
        "id": "NC-01",
        "category": "numeric_categorical",
        "query": "Show images from the Porous_Sponge category with a pixel size of approximately 100 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    },
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    },
                ],
            }
        },
    },
    {
        "id": "NC-02",
        "category": "numeric_categorical",
        "query": "Find images from the Nanowires category with an accelerating voltage between 5 kV and 10 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    },
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    },
                ],
            }
        },
    },
]

#### Three Numeric Fields

In [ ]:
THREE_NUMERIC_CASES = [
    {
        "id": "NNN-01",
        "category": "multiple_numeric",
        "query": "Show images with a pixel size of approximately 100 nm, a magnification of at least 500x, and a working distance of at most 15 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    },
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 500.0,
                    },
                    {
                        "field": "meta.working_distance_mm",
                        "operator": "<=",
                        "value": 15.0,
                    },
                ],
            }
        },
    },
    {
        "id": "NNN-02",
        "category": "multiple_numeric",
        "query": "Find images with an accelerating voltage between 5 kV and 10 kV, a brightness of at least 50%, and a contrast of at most 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    },
                    {
                        "field": "meta.brightness_percent",
                        "operator": ">=",
                        "value": 50.0,
                    },
                    {
                        "field": "meta.contrast_percent",
                        "operator": "<=",
                        "value": 75.0,
                    },
                ],
            }
        },
    },
]

#### Two Numeric + Categorical

In [ ]:
TWO_NUMERIC_CATEGORICAL_CASES = [
    {
        "id": "NNC-01",
        "category": "multiple_numeric_categorical",
        "query": "Show images from the Porous_Sponge category with a pixel size of approximately 100 nm and a magnification of at least 500x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    },
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    },
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 500.0,
                    },
                ],
            }
        },
    },
    {
        "id": "NNC-02",
        "category": "multiple_numeric_categorical",
        "query": "Find images from the Nanowires category with an accelerating voltage between 5 kV and 10 kV and a working distance of at most 15 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    },
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    },
                    {
                        "field": "meta.working_distance_mm",
                        "operator": "<=",
                        "value": 15.0,
                    },
                ],
            }
        },
    },
]

#### Two Categorical + Numeric

In [ ]:
TWO_CATEGORICAL_NUMERIC_CASES = [
    {
        "id": "CCN-01",
        "category": "multiple_categorical_numeric",
        "query": "Show images from the Porous_Sponge category taken with the InLens detector and with a pixel size of approximately 100 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    },
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    },
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    },
                ],
            }
        },
    },
    {
        "id": "CCN-02",
        "category": "multiple_categorical_numeric",
        "query": "Find images from the Nanowires category using the SE2 detector with a magnification of at least 1000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    },
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "SE2",
                    },
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 1000.0,
                    },
                ],
            }
        },
    },
]

### Non-Existing Metadata

#### Single Non-Existing Metadata

In [ ]:
NON_EXISTING_METADATA_CASES = [
    {
        "id": "NEM-01",
        "category": "non_existing",
        "query": "Show images with a detector angle of 45 degrees.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-02",
        "category": "non_existing",
        "query": "Find images with a sample temperature of 25 degrees.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-03",
        "category": "non_existing",
        "query": "Show images with an emission current of 10 microamps.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-04",
        "category": "non_existing",
        "query": "Find images with a chamber pressure of approximately 1 mbar.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-05",
        "category": "non_existing",
        "query": "Show images with an aperture size of 50 micrometers.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-06",
        "category": "non_existing",
        "query": "Find images with a stage tilt of at least 20 degrees.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-07",
        "category": "non_existing",
        "query": "Show images with a probe current between 1 and 5 nA.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-08",
        "category": "non_existing",
        "query": "Find images with a scan rotation of at most 30 degrees.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-09",
        "category": "non_existing",
        "query": "Show images with a detector distance of approximately 10 mm.",
        "expected": {
            "filters": None
        },
    },
    {
        "id": "NEM-10",
        "category": "non_existing",
        "query": "Find images with a sample thickness greater than or equal to 100 nm.",
        "expected": {
            "filters": None
        },
    },
]

#### Non-Existing Metadata with Multiple Constraints

In [ ]:
MULTIPLE_NON_EXISTING_METADATA_CASES = [
    {
        "id": "NEM-M01",
        "category": "non_existing_multiple",
        "query": (
            "Show images with a pixel size of approximately 100 nm and a detector angle of 45 degrees."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "NEM-M02",
        "category": "non_existing_multiple",
        "query": (
            "Find images with a magnification of at least 500x and a sample temperature of 25 degrees."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 500,
                    }
                ],
            }
        },
    },
    {
        "id": "NEM-M03",
        "category": "non_existing_multiple",
        "query": (
            "Show images with an accelerating voltage between 5 and 10 kV and an aperture size of 50 micrometers."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "NEM-M04",
        "category": "non_existing_multiple",
        "query": (
            "Find Porous_Sponge images taken with the InLens detector and a chamber pressure of approximately 1 mbar."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    },
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    },
                ],
            }
        },
    },
    {
        "id": "NEM-M05",
        "category": "non_existing_multiple",
        "query": (
            "Show images with a working distance of at most 15 mm, a brightness of at least 50%, and a detector distance of approximately 10 mm."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.working_distance_mm",
                        "operator": "<=",
                        "value": 15,
                    },
                    {
                        "field": "meta.brightness_percent",
                        "operator": ">=",
                        "value": 50,
                    },
                ],
            }
        },
    },
    {
        "id": "NEM-M06",
        "category": "non_existing_multiple",
        "query": (
            "Find Nanowires images with a scan speed setting between 5 and 10 and a sample thickness of at least 100 nm."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    },
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    },
                ],
            }
        },
    },
    {
        "id": "NEM-M07",
        "category": "non_existing_multiple",
        "query": (
            "Show images with a frame average count of at least 10, a line average count of at most 20, and a stage tilt of at least 20 degrees."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.frame_average_count",
                        "operator": ">=",
                        "value": 10,
                    },
                    {
                        "field": "meta.line_average_count",
                        "operator": "<=",
                        "value": 20,
                    },
                ],
            }
        },
    },
    {
        "id": "NEM-M08",
        "category": "non_existing_multiple",
        "query": (
            "Find Films_Coated_Surface images with a contrast of approximately 75% and an emission current of 10 microamps."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Films_Coated_Surface",
                    },
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.contrast_percent",
                                "operator": ">=",
                                "value": 67.5,
                            },
                            {
                                "field": "meta.contrast_percent",
                                "operator": "<=",
                                "value": 82.5,
                            },
                        ],
                    },
                ],
            }
        },
    },
    {
        "id": "NEM-M09",
        "category": "non_existing_multiple",
        "query": (
            "Show images with a magnification of at least 1000x, a pixel size of at most 50 nm, and a probe current between 1 and 5 nA."
        ),
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 1000,
                    },
                    {
                        "field": "meta.pixel_size_nm",
                        "operator": "<=",
                        "value": 50,
                    },
                ],
            }
        },
    },
    {
        "id": "NEM-M10",
        "category": "non_existing_multiple",
        "query": (
            "Find images with a detector angle of 45 degrees and a sample temperature of 25 degrees."
        ),
        "expected": {
            "filters": None
        },
    },
]

## Dictionary and Alias Checks

### Numeric Fields

In [ ]:
DICTIONARY_NUMERIC_CASES = [
    # __________ pixel_size_nm __________
    {
        "id": "DPIX-01",
        "category": "dictionary_numeric",
        "query": "Find images with an nm per pixel value of approximately 100.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 90.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 110.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DPIX-02",
        "category": "dictionary_numeric",
        "query": "Show images with a spatial sampling of at least 50 nm per pixel.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.pixel_size_nm",
                        "operator": ">=",
                        "value": 50,
                    }
                ],
            }
        },
    },
    {
        "id": "DPIX-03",
        "category": "dictionary_numeric",
        "query": "Find images with an image pixel size between 50 and 100 nm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": ">=",
                                "value": 50.0,
                            },
                            {
                                "field": "meta.pixel_size_nm",
                                "operator": "<=",
                                "value": 100.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ magnification_x __________
    {
        "id": "DMAG-01",
        "category": "dictionary_numeric",
        "query": "Show images acquired at a mag of approximately 500x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.magnification_x",
                                "operator": ">=",
                                "value": 450.0,
                            },
                            {
                                "field": "meta.magnification_x",
                                "operator": "<=",
                                "value": 550.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DMAG-02",
        "category": "dictionary_numeric",
        "query": "Find images with a magnification factor of at least 5000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.magnification_x",
                        "operator": ">=",
                        "value": 5000,
                    }
                ],
            }
        },
    },
    {
        "id": "DMAG-03",
        "category": "dictionary_numeric",
        "query": "Show images with a zoom level between 500x and 1000x.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.magnification_x",
                                "operator": ">=",
                                "value": 500.0,
                            },
                            {
                                "field": "meta.magnification_x",
                                "operator": "<=",
                                "value": 1000.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ accelerating_voltage_kv __________
    {
        "id": "DAV-01",
        "category": "dictionary_numeric",
        "query": "Find images acquired with an EHT of approximately 10 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DAV-02",
        "category": "dictionary_numeric",
        "query": "Show images with a beam voltage of at least 15 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.accelerating_voltage_kv",
                        "operator": ">=",
                        "value": 15,
                    }
                ],
            }
        },
    },
    {
        "id": "DAV-03",
        "category": "dictionary_numeric",
        "query": "Find images with a high tension between 5 and 10 kV.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.accelerating_voltage_kv",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ working_distance_mm __________
    {
        "id": "DWD-01",
        "category": "dictionary_numeric",
        "query": "Show images with a WD of approximately 10 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.working_distance_mm",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.working_distance_mm",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DWD-02",
        "category": "dictionary_numeric",
        "query": "Find images with a focus distance of at most 5 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.working_distance_mm",
                        "operator": "<=",
                        "value": 5,
                    }
                ],
            }
        },
    },
    {
        "id": "DWD-03",
        "category": "dictionary_numeric",
        "query": "Show images with a specimen distance between 5 and 10 mm.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.working_distance_mm",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.working_distance_mm",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ brightness_percent __________
    {
        "id": "DBRI-01",
        "category": "dictionary_numeric",
        "query": "Find images with an image brightness of approximately 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.brightness_percent",
                                "operator": ">=",
                                "value": 45.0,
                            },
                            {
                                "field": "meta.brightness_percent",
                                "operator": "<=",
                                "value": 55.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DBRI-02",
        "category": "dictionary_numeric",
        "query": "Show images with a brightness level of at least 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.brightness_percent",
                        "operator": ">=",
                        "value": 75,
                    }
                ],
            }
        },
    },
    {
        "id": "DBRI-03",
        "category": "dictionary_numeric",
        "query": "Find images with a brightness setting between 25% and 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.brightness_percent",
                                "operator": ">=",
                                "value": 25.0,
                            },
                            {
                                "field": "meta.brightness_percent",
                                "operator": "<=",
                                "value": 50.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ contrast_percent __________
    {
        "id": "DCON-01",
        "category": "dictionary_numeric",
        "query": "Show images with an image contrast of approximately 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.contrast_percent",
                                "operator": ">=",
                                "value": 45.0,
                            },
                            {
                                "field": "meta.contrast_percent",
                                "operator": "<=",
                                "value": 55.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DCON-02",
        "category": "dictionary_numeric",
        "query": "Find images with a dynamic range of at least 75%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.contrast_percent",
                        "operator": ">=",
                        "value": 75,
                    }
                ],
            }
        },
    },
    {
        "id": "DCON-03",
        "category": "dictionary_numeric",
        "query": "Show images with an intensity contrast between 25% and 50%.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.contrast_percent",
                                "operator": ">=",
                                "value": 25.0,
                            },
                            {
                                "field": "meta.contrast_percent",
                                "operator": "<=",
                                "value": 50.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ scan_speed_setting __________
    {
        "id": "DSS-01",
        "category": "dictionary_numeric",
        "query": "Find images acquired at a scan rate of approximately 5.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": ">=",
                                "value": 4.5,
                            },
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": "<=",
                                "value": 5.5,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DSS-02",
        "category": "dictionary_numeric",
        "query": "Show images with a scanning speed of at least 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.scan_speed_setting",
                        "operator": ">=",
                        "value": 10,
                    }
                ],
            }
        },
    },
    {
        "id": "DSS-03",
        "category": "dictionary_numeric",
        "query": "Find images with an acquisition speed between 5 and 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.scan_speed_setting",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ frame_average_count __________
    {
        "id": "DFA-01",
        "category": "dictionary_numeric",
        "query": "Show images with approximately 10 frames averaged.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.frame_average_count",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.frame_average_count",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DFA-02",
        "category": "dictionary_numeric",
        "query": "Find images using a frame average count of at least 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.frame_average_count",
                        "operator": ">=",
                        "value": 20,
                    }
                ],
            }
        },
    },
    {
        "id": "DFA-03",
        "category": "dictionary_numeric",
        "query": "Show images with the number of frames averaged between 5 and 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.frame_average_count",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.frame_average_count",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },

    # __________ line_average_count __________
    {
        "id": "DLA-01",
        "category": "dictionary_numeric",
        "query": "Find images with approximately 10 lines averaged.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.line_average_count",
                                "operator": ">=",
                                "value": 9.0,
                            },
                            {
                                "field": "meta.line_average_count",
                                "operator": "<=",
                                "value": 11.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
    {
        "id": "DLA-02",
        "category": "dictionary_numeric",
        "query": "Show images with a line average count of at least 20.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.line_average_count",
                        "operator": ">=",
                        "value": 20,
                    }
                ],
            }
        },
    },
    {
        "id": "DLA-03",
        "category": "dictionary_numeric",
        "query": "Find images with the number of lines averaged between 5 and 10.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "operator": "AND",
                        "conditions": [
                            {
                                "field": "meta.line_average_count",
                                "operator": ">=",
                                "value": 5.0,
                            },
                            {
                                "field": "meta.line_average_count",
                                "operator": "<=",
                                "value": 10.0,
                            },
                        ],
                    }
                ],
            }
        },
    },
]

### Categorical Fields

In [ ]:
DICTIONARY_CATEGORICAL_CASES = [
    # __________ category — Porous_Sponge __________
    {
        "id": "DCAT-01",
        "category": "dictionary_categorical",
        "query": "Find images of porous sponges.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-02",
        "category": "dictionary_categorical",
        "query": "Show images with a sponge-like morphology.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-03",
        "category": "dictionary_categorical",
        "query": "Find images with a sponge like structure.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Porous_Sponge",
                    }
                ],
            }
        },
    },

    # __________ category — Patterned_surface __________
    {
        "id": "DCAT-04",
        "category": "dictionary_categorical",
        "query": "Find images of patterned surfaces.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Patterned_surface",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-05",
        "category": "dictionary_categorical",
        "query": "Show images with a surface pattern.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Patterned_surface",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-06",
        "category": "dictionary_categorical",
        "query": "Find patterned images.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Patterned_surface",
                    }
                ],
            }
        },
    },

    # __________ category — Particles __________
    {
        "id": "DCAT-07",
        "category": "dictionary_categorical",
        "query": "Find particle images.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Particles",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-08",
        "category": "dictionary_categorical",
        "query": "Show images containing nanoparticles.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Particles",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-09",
        "category": "dictionary_categorical",
        "query": "Find images of nano particles.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Particles",
                    }
                ],
            }
        },
    },

    # __________ category — Films_Coated_Surface __________
    {
        "id": "DCAT-10",
        "category": "dictionary_categorical",
        "query": "Find images of films.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Films_Coated_Surface",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-11",
        "category": "dictionary_categorical",
        "query": "Show images of coated surfaces.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Films_Coated_Surface",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-12",
        "category": "dictionary_categorical",
        "query": "Find images with a thin film.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Films_Coated_Surface",
                    }
                ],
            }
        },
    },

    # __________ category — Powder __________
    {
        "id": "DCAT-13",
        "category": "dictionary_categorical",
        "query": "Find images of powders.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Powder",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-14",
        "category": "dictionary_categorical",
        "query": "Show images of a powder sample.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Powder",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-15",
        "category": "dictionary_categorical",
        "query": "Find images of powder samples.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Powder",
                    }
                ],
            }
        },
    },

    # __________ category — Tips __________
    {
        "id": "DCAT-16",
        "category": "dictionary_categorical",
        "query": "Find images of tips.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Tips",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-17",
        "category": "dictionary_categorical",
        "query": "Show images of AFM tips.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Tips",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-18",
        "category": "dictionary_categorical",
        "query": "Find images of probe tips.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Tips",
                    }
                ],
            }
        },
    },

    # __________ category — Nanowires __________
    {
        "id": "DCAT-19",
        "category": "dictionary_categorical",
        "query": "Find images of nanowires.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-20",
        "category": "dictionary_categorical",
        "query": "Show images of nano wires.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-21",
        "category": "dictionary_categorical",
        "query": "Find a nanowire sample.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Nanowires",
                    }
                ],
            }
        },
    },

    # __________ category — Biological __________
    {
        "id": "DCAT-22",
        "category": "dictionary_categorical",
        "query": "Find biological samples.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Biological",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-23",
        "category": "dictionary_categorical",
        "query": "Show images of a bio sample.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Biological",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-24",
        "category": "dictionary_categorical",
        "query": "Find images of bio samples.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Biological",
                    }
                ],
            }
        },
    },

    # __________ category — MEMS_devices_and_electrodes __________
    {
        "id": "DCAT-25",
        "category": "dictionary_categorical",
        "query": "Find images of MEMS devices.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "MEMS_devices_and_electrodes",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-26",
        "category": "dictionary_categorical",
        "query": "Show images of electrodes.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "MEMS_devices_and_electrodes",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-27",
        "category": "dictionary_categorical",
        "query": "Find microelectromechanical systems.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "MEMS_devices_and_electrodes",
                    }
                ],
            }
        },
    },

    # __________ category — Fibres __________
    {
        "id": "DCAT-28",
        "category": "dictionary_categorical",
        "query": "Find images of fibers.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Fibres",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-29",
        "category": "dictionary_categorical",
        "query": "Show images of fibrous structures.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Fibres",
                    }
                ],
            }
        },
    },
    {
        "id": "DCAT-30",
        "category": "dictionary_categorical",
        "query": "Find images of fibres.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.category",
                        "operator": "==",
                        "value": "Fibres",
                    }
                ],
            }
        },
    },

    # __________ detector — InLens __________
    {
        "id": "DDET-01",
        "category": "dictionary_categorical",
        "query": "Find images acquired with an in lens detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    }
                ],
            }
        },
    },
    {
        "id": "DDET-02",
        "category": "dictionary_categorical",
        "query": "Show images using an in-lens detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    }
                ],
            }
        },
    },
    {
        "id": "DDET-03",
        "category": "dictionary_categorical",
        "query": "Find images taken with the inlens detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "InLens",
                    }
                ],
            }
        },
    },

    # __________ detector — SE2 __________
    {
        "id": "DDET-04",
        "category": "dictionary_categorical",
        "query": "Find images acquired with a secondary electron detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "SE2",
                    }
                ],
            }
        },
    },
    {
        "id": "DDET-05",
        "category": "dictionary_categorical",
        "query": "Show images using secondary electrons.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "SE2",
                    }
                ],
            }
        },
    },
    {
        "id": "DDET-06",
        "category": "dictionary_categorical",
        "query": "Find images acquired with an SE detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "SE2",
                    }
                ],
            }
        },
    },

    # __________ detector — HDAsB __________
    {
        "id": "DDET-07",
        "category": "dictionary_categorical",
        "query": "Find images acquired with a backscatter detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "HDAsB",
                    }
                ],
            }
        },
    },
    {
        "id": "DDET-08",
        "category": "dictionary_categorical",
        "query": "Show images using a BSE detector.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "HDAsB",
                    }
                ],
            }
        },
    },
    {
        "id": "DDET-09",
        "category": "dictionary_categorical",
        "query": "Find images with backscattered electrons.",
        "expected": {
            "filters": {
                "operator": "AND",
                "conditions": [
                    {
                        "field": "meta.detector",
                        "operator": "==",
                        "value": "HDAsB",
                    }
                ],
            }
        },
    },
]

# Metadata Extraction

## Metadata Schema

In [ ]:
METADATA_SCHEMA = {
    # __________ Numeric fields __________
    "pixel_size_nm": {
        "type": "numeric",
        "description": (
            "Physical spatial sampling represented by one image pixel. "
            "It expresses the real-space scale of the SEM image and allows "
            "images acquired at different spatial scales to be compared."
        ),
        "dictionary_terms": [
            "pixel size",
            "image pixel size",
            "pixel dimension",
            "nm per pixel",
            "nm/pixel",
            "nanometers per pixel",
            "nanometres per pixel",
            "um per pixel",
            "µm per pixel",
            "micrometers per pixel",
            "micrometres per pixel",
            "spatial sampling",
            "pixel sampling",
            "sampling resolution",
        ],
        "unit": "nm/pixel",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "magnification_x": {
        "type": "numeric",
        "description": (
            "Nominal SEM magnification used when the image was acquired."
        ),
        "dictionary_terms": [
            "magnification",
            "mag",
            "magnified",
            "magnification factor",
            "times magnification",
            "x magnification",
            "kx",
            "KX",
            "zoom level",
        ],
        "unit": "x",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "accelerating_voltage_kv": {
        "type": "numeric",
        "description": (
            "Electron beam accelerating voltage, commonly reported as EHT. "
            "It influences electron penetration, interaction volume, "
            "surface sensitivity, charging behaviour and signal generation."
        ),
        "dictionary_terms": [
            "accelerating voltage",
            "acceleration voltage",
            "EHT",
            "beam voltage",
            "electron beam voltage",
            "electron voltage",
            "high tension",
            "high voltage",
            "operating voltage",
            "kV",
            "kilovolts",
            "kilovolt",
        ],
        "unit": "kV",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "working_distance_mm": {
        "type": "numeric",
        "description": (
            "SEM working distance associated with the specimen-to-objective "
            "imaging geometry. It influences imaging conditions such as depth "
            "of field, resolution and detector geometry."
        ),
        "dictionary_terms": [
            "working distance",
            "WD",
            "work distance",
            "working dist",
            "working-distance",
            "focus distance",
            "sample distance",
            "specimen distance",
        ],
        "unit": "mm",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "brightness_percent": {
        "type": "numeric",
        "description": (
            "Instrument brightness setting applied during acquisition. "
            "It controls the displayed or saved signal intensity level."
        ),
        "dictionary_terms": [
            "brightness",
            "brightness level",
            "image brightness",
            "brightness setting",
            "brightness percentage",
            "bright",
            "intensity level",
        ],
        "unit": "%",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "contrast_percent": {
        "type": "numeric",
        "description": (
            "Instrument contrast setting applied during acquisition. "
            "It controls the separation and dynamic range of intensities "
            "represented in the SEM image."
        ),
        "dictionary_terms": [
            "contrast",
            "contrast level",
            "image contrast",
            "contrast setting",
            "contrast percentage",
            "dynamic range",
            "intensity contrast",
        ],
        "unit": "%",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "scan_speed_setting": {
        "type": "numeric",
        "description": (
            "Instrument-specific scan speed setting controlling how rapidly "
            "the electron beam scans the image. It is an ordinal acquisition "
            "setting rather than a physical time unless separately calibrated."
        ),
        "dictionary_terms": [
            "scan speed",
            "scan rate",
            "scanrate",
            "scanning speed",
            "scanning rate",
            "scan speed setting",
            "scan setting",
            "acquisition speed",
            "raster speed",
        ],
        "unit": "setting",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "frame_average_count": {
        "type": "numeric",
        "description": (
            "Number of image frames averaged together during acquisition "
            "to reduce random noise and improve signal-to-noise ratio."
        ),
        "dictionary_terms": [
            "frame averaging",
            "frame average",
            "frame average count",
            "frame averaging count",
            "frames averaged",
            "number of frames averaged",
            "frames to average",
            "average frames",
            "frame count",
        ],
        "unit": "count",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "line_average_count": {
        "type": "numeric",
        "description": (
            "Number of repeated scan lines averaged during acquisition "
            "to reduce noise and improve signal stability."
        ),
        "dictionary_terms": [
            "line averaging",
            "line average",
            "line average count",
            "line averaging count",
            "lines averaged",
            "number of lines averaged",
            "lines to average",
            "average lines",
        ],
        "unit": "count",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    # __________ Categorical fields __________
    "category": {
        "type": "categorical",
        "description": (
            "Human-assigned SEM morphology category from the annotated "
            "dataset described by Aversa et al. The categories were defined "
            "mainly from the dominant shape, structure and dimensionality "
            "of the imaged object."
        ),
        "values": {
            "Porous_Sponge": {
                "description": (
                    "SEM images assigned to the porous sponge category, "
                    "characterized by a porous or sponge-like morphology."
                ),
                "dictionary_terms": [
                    "porous sponge",
                    "porous sponges",
                    "sponge-like",
                    "sponge like",
                ],
            },
            "Patterned_surface": {
                "description": (
                    "SEM images of surfaces containing organized, repeated "
                    "or lithographically defined patterns or structures."
                ),
                "dictionary_terms": [
                    "patterned surface",
                    "patterned surfaces",
                    "surface pattern",
                    "surface patterns",
                    "patterned",
                ],
            },
            "Particles": {
                "description": (
                    "Zero-dimensional particle-like objects that may be "
                    "individually dispersed or grouped into clustered structures."
                ),
                "dictionary_terms": [
                    "particle",
                    "particles",
                    "nanoparticle",
                    "nanoparticles",
                    "nano particle",
                    "nano particles",
                ],
            },
            "Films_Coated_Surface": {
                "description": (
                    "Two-dimensional films or coatings on a surface, including "
                    "surfaces with different degrees of roughness or particle coverage."
                ),
                "dictionary_terms": [
                    "film",
                    "films",
                    "coated surface",
                    "coated surfaces",
                    "coating",
                    "coatings",
                    "thin film",
                    "thin films",
                ],
            },
            "Powder": {
                "description": (
                    "SEM images assigned to the powder category in the annotated dataset."
                ),
                "dictionary_terms": [
                    "powder",
                    "powders",
                    "powder sample",
                    "powder samples",
                ],
            },
            "Tips": {
                "description": (
                    "Tip-like structures, including images commonly associated "
                    "with atomic force microscopy probe tips."
                ),
                "dictionary_terms": [
                    "tip",
                    "tips",
                    "AFM tip",
                    "AFM tips",
                    "probe tip",
                    "probe tips",
                ],
            },
            "Nanowires": {
                "description": (
                    "One-dimensional nanowire structures. In the dataset hierarchy, "
                    "nanowires can include individual, few, entangled or forest-like arrangements."
                ),
                "dictionary_terms": [
                    "nanowire",
                    "nanowires",
                    "nano wire",
                    "nano wires",
                ],
            },
            "Biological": {
                "description": (
                    "SEM images of biological samples grouped into the Biological "
                    "category of the annotated dataset."
                ),
                "dictionary_terms": [
                    "biological",
                    "biological sample",
                    "biological samples",
                    "bio sample",
                    "bio samples",
                ],
            },
            "MEMS_devices_and_electrodes": {
                "description": (
                    "Microelectromechanical devices, electrodes and related "
                    "microfabricated structures, typically associated with lithographic fabrication."
                ),
                "dictionary_terms": [
                    "MEMS",
                    "MEMS device",
                    "MEMS devices",
                    "electrode",
                    "electrodes",
                    "microelectromechanical",
                    "microelectromechanical system",
                    "microelectromechanical systems",
                    "microdevice",
                    "microdevices",
                ],
            },
            "Fibres": {
                "description": (
                    "One-dimensional fibrous structures assigned to the dedicated "
                    "Fibres category of the annotated dataset."
                ),
                "dictionary_terms": [
                    "fibre",
                    "fibres",
                    "fiber",
                    "fibers",
                    "fibrous",
                ],
            },
        },
    },

    "detector": {
        "type": "categorical",
        "description": (
            "Detector or signal channel used to form the SEM image. "
            "Detector choice affects the type of electron signal recorded "
            "and therefore the appearance and scientific information represented in the image."
        ),
        "values": {
            "InLens": {
                "description": (
                    "In-lens detector signal used for SEM image formation. "
                    "The instrument described in the dataset paper includes "
                    "a high-efficiency in-lens detector."
                ),
                "dictionary_terms": [
                    "InLens",
                    "in lens",
                    "in-lens",
                    "inlens",
                    "in lens detector",
                    "in-lens detector",
                ],
            },
            "SE2": {
                "description": (
                    "Secondary-electron detector channel labelled SE2 "
                    "in the instrument metadata."
                ),
                "dictionary_terms": [
                    "SE2",
                    "se2",
                    "secondary electron",
                    "secondary electrons",
                    "secondary electron detector",
                    "SE detector",
                ],
            },
            "HDAsB": {
                "description": (
                    "Backscattered-electron detector channel labelled HDAsB "
                    "in the instrument metadata."
                ),
                "dictionary_terms": [
                    "HDAsB",
                    "hdasb",
                    "backscatter detector",
                    "backscattered electron detector",
                    "backscattered electrons",
                    "BSE detector",
                    "BSE",
                ],
            },
        },
    },
}

In [ ]:
def build_metadata_schema_description() -> str:
    """
    Convert the internal metadata schema into JSON for the metadata
    extraction LLM.

    The schema contains all information the LLM needs to map natural
    language to canonical metadata fields and values.
    """
    return json.dumps(METADATA_SCHEMA, indent=2, ensure_ascii=False)

METADATA_SCHEMA_DESCRIPTION = build_metadata_schema_description()

## Metadata Extractor

In [ ]:
@component
class QueryMetadataExtractor():
    """
    Extract metadata constraints from a natural-language query.

    The LLM is responsible for mapping the user's wording to canonical
    metadata fields and categorical values.

    Python is responsible for validating the result and converting it
    into a Qdrant filter.
    """
    def __init__(self):
        template = [
            ChatMessage.from_system(
                """
                You are part of an information system that extracts SEM metadata
                constraints from natural-language user queries.

                Your task is to identify metadata constraints explicitly provided by
                the user and map them to the canonical metadata fields and values
                defined in the supplied metadata schema.

                The metadata schema contains:
                - canonical field names
                - field types
                - scientific descriptions
                - alternative/dictionary terms
                - units for numeric fields
                - allowed values for categorical fields
                - alternative terms and descriptions for categorical values

                Use the scientific descriptions and dictionary terms to understand
                what the user means. The user does not need to use the canonical
                field name.

                Do NOT use general SEM knowledge to infer metadata that the user
                did not provide.

                ------------------------------------------------------------
                REQUIRED OUTPUT FORMAT
                ------------------------------------------------------------

                The output MUST always be a single JSON object.

                Each metadata field MUST be a top-level key in the JSON object.

                The value associated with each field MUST be an object containing:
                - "operator"
                - "value"

                The canonical metadata field name MUST be used as the top-level key.

                The keys "operator" and "value" MUST NEVER appear directly at
                the top level of the JSON object.

                For example, if the canonical field is "category", the correct
                structure is:

                {
                    "category": {
                        "operator": "==",
                        "value": "Porous_Sponge"
                    }
                }

                The following structure is INVALID:

                {
                    "operator": "==",
                    "value": "Porous_Sponge"
                }

                ------------------------------------------------------------
                NUMERIC FIELDS
                ------------------------------------------------------------

                For numeric fields, return:

                {
                    "<canonical_field_name>": {
                        "operator": "<operator>",
                        "value": <numeric value>
                    }
                }

                Supported operators:

                ">=": Greater than or equal to the specified value.

                "<=": Less than or equal to the specified value.

                "~": Approximately equal to the specified value.

                Use "~" when the user says:
                - around
                - about
                - roughly
                - approximately

                Also use "~" when the user gives a numeric value that sounds
                like an exact desired value without specifying an explicit
                comparison or range.

                "between": Use this when the user explicitly gives a range with two bounds.

                    The value MUST be an array:

                    [low, high]

                Examples:

                User:
                "images with pixel size around 100 nm"

                Output:

                {
                    "pixel_size_nm": {
                        "operator": "~",
                        "value": 100
                    }
                }

                User:
                "images with pixel size greater than 100 nm"

                Output:

                {
                    "pixel_size_nm": {
                        "operator": ">=",
                        "value": 100
                    }
                }

                User:
                "images with pixel size up to 100 nm"

                Output:

                {
                    "pixel_size_nm": {
                        "operator": "<=",
                        "value": 100
                    }
                }

                User:
                "images with pixel size between 80 and 120 nm"

                Output:

                {
                    "pixel_size_nm": {
                        "operator": "between",
                        "value": [80, 120]
                    }
                }

                Do not convert units unless the query explicitly provides a unit
                that can be safely interpreted.

                If the user provides a unit, use the canonical numeric field
                and represent the numeric value consistently with the metadata
                schema.

                ------------------------------------------------------------
                CATEGORICAL FIELDS
                ------------------------------------------------------------

                For categorical fields, map the user's wording to exactly one
                of the allowed canonical values defined in the metadata schema.

                The canonical categorical value MUST be used.

                The categorical field MUST be the top-level JSON key.

                The structure MUST be:

                {
                    "<canonical_field_name>": {
                        "operator": "==",
                        "value": "<allowed canonical value>"
                    }
                }

                Never return a categorical value that is not present in the
                allowed values defined by the schema.

                Never invent a categorical value.

                Example:

                Suppose the schema contains:

                category:
                    Porous_Sponge:
                        dictionary terms:
                            porous sponge
                            porous sponges
                            sponge
                            sponge-like structure

                Then the query:

                "show SEM images that are Porous Sponges"

                MUST produce:

                {
                    "category": {
                        "operator": "==",
                        "value": "Porous_Sponge"
                    }
                }

                It MUST NOT produce:

                {
                    "operator": "==",
                    "value": "Porous_Sponge"
                }

                Another example:

                Suppose the schema contains:

                detector:
                    SE2:
                        dictionary terms:
                            secondary electron
                            secondary electron detector
                            SE detector

                Then:

                "images acquired with a secondary electron detector"

                MUST produce:

                {
                    "detector": {
                        "operator": "==",
                        "value": "SE2"
                    }
                }

                ------------------------------------------------------------
                MULTIPLE METADATA CONSTRAINTS
                ------------------------------------------------------------

                If the user provides multiple metadata constraints, include all
                of them as separate top-level fields in the same JSON object.

                For example:

                User:
                "show SEM images that are Porous Sponges with 100 nm pixel size"

                Output:

                {
                    "category": {
                        "operator": "==",
                        "value": "Porous_Sponge"
                    },
                    "pixel_size_nm": {
                        "operator": "~",
                        "value": 100
                    }
                }

                Another example:

                User:
                "show Porous Sponge images acquired at 10 kV with a 100 nm pixel size"

                Output:

                {
                    "category": {
                        "operator": "==",
                        "value": "Porous_Sponge"
                    },
                    "accelerating_voltage_kv": {
                        "operator": "~",
                        "value": 10
                    },
                    "pixel_size_nm": {
                        "operator": "~",
                        "value": 100
                    }
                }

                Each constraint MUST have its own canonical field name.

                Do not combine multiple constraints into a single object.

                ------------------------------------------------------------
                GENERAL RULES
                ------------------------------------------------------------

                1. Only include a metadata field if the query explicitly provides
                or clearly implies its value.

                2. Do not infer metadata from general SEM knowledge.

                3. Do not infer metadata from the morphology described by the user.

                4. Do not invent canonical field names.

                5. Do not invent categorical values.

                6. For categorical fields, the value MUST exactly match one of the
                allowed canonical values in the supplied schema.

                7. For numeric fields, use only the supported operators:
                >=, <=, ~, between.

                8. For "between", the value MUST contain exactly two numeric values:
                [low, high]

                9. Preserve the meaning of the user's constraint. Do not silently
                change a comparison into another comparison.

                10. If the user does not specify a metadata constraint, do not
                    invent one.

                11. If no metadata fields are present in the query, return exactly:

                {}

                12. The output must contain ONLY valid JSON.

                13. Do not include Markdown code fences.

                14. Do not include explanations, comments, or additional text.

                15. The top-level JSON object MUST contain canonical metadata field
                    names, never "operator" or "value".

                ------------------------------------------------------------
                FINAL EXAMPLES
                ------------------------------------------------------------

                Query:
                "Find spherical particles."

                Output:

                {}

                Query:
                "show images that are Porous Sponges"

                Output:

                {
                    "category": {
                        "operator": "==",
                        "value": "Porous_Sponge"
                    }
                }

                Query:
                "show images with 100 nm pixel size"

                Output:

                {
                    "pixel_size_nm": {
                        "operator": "~",
                        "value": 100
                    }
                }

                Query:
                "show Porous Sponge images with 100 nm pixel size"

                Output:

                {
                    "category": {
                        "operator": "==",
                        "value": "Porous_Sponge"
                    },
                    "pixel_size_nm": {
                        "operator": "~",
                        "value": 100
                    }
                }

                Query:
                "show images with pixel size between 50 and 100 nm and category Powder"

                Output:

                {
                    "pixel_size_nm": {
                        "operator": "between",
                        "value": [50, 100]
                    },
                    "category": {
                        "operator": "==",
                        "value": "Powder"
                    }
                }

                Remember:

                - Top-level keys = canonical metadata field names.
                - Nested "operator" = comparison operator.
                - Nested "value" = extracted value.
                - Never put "operator" or "value" directly at the top level.
                """
            ),
            ChatMessage.from_user(
                """
                Query:
                {{ query }}

                Metadata schema:
                {{ metadata_schema }}

                Extracted metadata:
                """
            ),
        ]

        self.pipeline = Pipeline()

        self.pipeline.add_component(
            "builder", 
            ChatPromptBuilder(
                template=template, 
                required_variables={"query", "metadata_schema"}
            )
        )
        self.pipeline.add_component(
            "llm", 
            VLLMChatGenerator(
                model=VLLM.GEMMA_MODEL, 
                api_key=Secret.from_token(VLLM.API_KEY), 
                api_base_url=VLLM.API_URL, 
                generation_kwargs={
                    "response_format": {"type": "json_object"},
                    # "extra_body": {"chat_template_kwargs": {"reasoning_effort": "none"}} # "none", "low", "medium", "high" ("xhigh" for Qwen)
                    "extra_body": {"chat_template_kwargs": {"enable_thinking": False}}
                }
            )
        )
        self.pipeline.connect("builder", "llm")

    def build_condition(self, field: str, extracted: dict):
        """
        Validate one extracted metadata field and convert it into
        a Qdrant filter condition.
        """
        schema = METADATA_SCHEMA[field]
        field_type = schema["type"]

        operator = extracted.get("operator")
        value = extracted.get("value")

        # __________ Categorical fields __________
        if field_type == "categorical":
            if operator != "==":
                return None

            if not isinstance(value, str):
                return None

            allowed_values = schema.get("values", {})

            if value not in allowed_values:
                return None

            return {
                "field": f"meta.{field}",
                "operator": "==",
                "value": value,
            }

        # __________ Numeric fields __________
        if field_type != "numeric":
            return None

        valid_numeric_operators = {
            ">=",
            "<=",
            "~",
            "between",
        }

        if operator not in valid_numeric_operators:
            return None

        # __________ Numeric range __________
        if operator == "between":
            if not isinstance(value, list) or len(value) != 2:
                return None

            try:
                low = float(value[0])
                high = float(value[1])
            except (TypeError, ValueError):
                return None

            if low > high:
                low, high = high, low

            return {
                "operator": "AND",
                "conditions": [
                    {
                        "field": f"meta.{field}",
                        "operator": ">=",
                        "value": low,
                    },
                    {
                        "field": f"meta.{field}",
                        "operator": "<=",
                        "value": high,
                    },
                ],
            }

        # __________ Numeric value __________
        try:
            value = float(value)
        except (TypeError, ValueError):
            return None

        # __________ Approximate numeric value __________
        if operator == "~":
            filter_config = schema.get("filter", {})

            tolerance = filter_config.get("tolerance", 0.10)

            tolerance_type = filter_config.get("tolerance_type", "relative")

            if tolerance_type == "relative":
                delta = abs(value) * tolerance
            else:
                delta = tolerance

            return {
                "operator": "AND",
                "conditions": [
                    {
                        "field": f"meta.{field}",
                        "operator": ">=",
                        "value": value - delta,
                    },
                    {
                        "field": f"meta.{field}",
                        "operator": "<=",
                        "value": value + delta,
                    },
                ],
            }

        # ">=", "<=" map directly onto Qdrant's own operator strings
        return {
            "field": f"meta.{field}",
            "operator": operator,
            "value": value,
        }

    @component.output_types(filters = dict[str, str])
    def run(self, query: str, metadata_schema: str):
        """
        Extract metadata from the query and convert it into a Qdrant
        filter.

        Invalid or hallucinated fields/values are silently discarded.
        If nothing valid remains, no filter is returned.
        """
        result = self.pipeline.run({'builder': {'query': query, 'metadata_schema': metadata_schema}})
        try:
            replies = result["llm"]["replies"]

            if not replies:
                return {"filters": None}

            metadata = json.loads(replies[0].text)

            if not isinstance(metadata, dict):
                return {"filters": None}

            conditions = []

            for field, extracted in metadata.items():

                # Prevent hallucinated field names.
                if field not in METADATA_SCHEMA:
                    continue

                # Each field must contain an extraction object.
                if not isinstance(extracted, dict):
                    continue

                condition = self.build_condition(field, extracted)

                if condition is not None:
                    conditions.append(condition)

            if not conditions:
                return {"filters": None}

            return {"filters": {"operator": "AND", "conditions": conditions}}

        except Exception as exc:
            return {"filters": None}

# Evaluation

In [ ]:
def normalize_filter(obj):
    if isinstance(obj, dict):
        normalized = {
            key: normalize_filter(value)
            for key, value in obj.items()
        }

        if "conditions" in normalized:
            normalized["conditions"] = sorted(
                normalized["conditions"],
                key=str,
            )

        return normalized

    if isinstance(obj, list):
        return [
            normalize_filter(value)
            for value in obj
        ]

    return obj

In [ ]:
def run_test_case(extractor, test_case):
    actual = extractor.run(
        query=test_case["query"],
        metadata_schema=METADATA_SCHEMA_DESCRIPTION,
    )

    return {
        "id": test_case["id"],
        "category": test_case["category"],
        "query": test_case["query"],
        "expected": test_case["expected"],
        "actual": actual,
        "correct": normalize_filter(actual) == normalize_filter(test_case["expected"]),
    }

In [ ]:
TEST_CASES = (
    PIXEL_SIZE_CASES
    + MAGNIFICATION_CASES
    + ACCELERATING_VOLTAGE_CASES
    + WORKING_DISTANCE_CASES
    + BRIGHTNESS_CASES
    + CONTRAST_CASES
    + SCAN_SPEED_CASES
    + FRAME_AVERAGE_COUNT_CASES
    + LINE_AVERAGE_COUNT_CASES
    + CATEGORY_CASES
    + DETECTOR_CASES
    + NUMERIC_NUMERIC_CASES
    + CATEGORICAL_CATEGORICAL_CASES
    + NUMERIC_CATEGORICAL_CASES
    + THREE_NUMERIC_CASES
    + TWO_NUMERIC_CATEGORICAL_CASES
    + TWO_CATEGORICAL_NUMERIC_CASES
    + NON_EXISTING_METADATA_CASES
    + MULTIPLE_NON_EXISTING_METADATA_CASES
    + DICTIONARY_NUMERIC_CASES
    + DICTIONARY_CATEGORICAL_CASES
)

print(f"Total test cases: {len(TEST_CASES)}")

In [ ]:
extractor = QueryMetadataExtractor()

In [ ]:
results = [run_test_case(extractor, test_case) for test_case in TEST_CASES]

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# Overall
print(f"Accuracy: {results_df['correct'].mean():.2%}")

# By category
print(
    results_df
    .groupby("category")["correct"]
    .mean()
    .sort_values()
)

In [ ]:
# Failures only
failures = results_df[~results_df["correct"]]

for _, row in failures.iterrows():
    print("=" * 100)
    print(f"{row['id']} | {row['category']}")
    print(f"QUERY:\n{row['query']}")
    print(f"EXPECTED:\n{row['expected']}")
    print(f"ACTUAL:\n{row['actual']}")

## No Thinking

### Gemma

```text
Accuracy: 100.00%
category
between                         1.0
categorical                     1.0
dictionary_categorical          1.0
dictionary_numeric              1.0
multiple_categorical            1.0
multiple_categorical_numeric    1.0
multiple_numeric                1.0
multiple_numeric_categorical    1.0
non_existing                    1.0
non_existing_multiple           1.0
numeric_approx                  1.0
numeric_between                 1.0
numeric_categorical             1.0
numeric_comparison              1.0
Name: correct, dtype: float64
Time: 1m 17.5s
```

### Qwen 3.6

```text
Accuracy: 98.47%
category
non_existing                    0.900000
non_existing_multiple           0.900000
dictionary_numeric              0.962963
between                         1.000000
categorical                     1.000000
dictionary_categorical          1.000000
multiple_categorical            1.000000
multiple_categorical_numeric    1.000000
multiple_numeric                1.000000
multiple_numeric_categorical    1.000000
numeric_approx                  1.000000
numeric_between                 1.000000
numeric_categorical             1.000000
numeric_comparison              1.000000
Name: correct, dtype: float64
Time: 1m 20.5s
```

*Errors:*
```text
====================================================================================================
NEM-09 | non_existing
QUERY:
Show images with a detector distance of approximately 10 mm.
EXPECTED:
{'filters': None}
ACTUAL:
{'filters': {'operator': 'AND', 'conditions': [{'operator': 'AND', 'conditions': [{'field': 'meta.working_distance_mm', 'operator': '>=', 'value': 9.0}, {'field': 'meta.working_distance_mm', 'operator': '<=', 'value': 11.0}]}]}}
====================================================================================================
NEM-M06 | non_existing_multiple
QUERY:
Find Nanowires images with a scan speed setting between 5 and 10 and a sample thickness of at least 100 nm.
EXPECTED:
{'filters': {'operator': 'AND', 'conditions': [{'field': 'meta.category', 'operator': '==', 'value': 'Nanowires'}, {'operator': 'AND', 'conditions': [{'field': 'meta.scan_speed_setting', 'operator': '>=', 'value': 5.0}, {'field': 'meta.scan_speed_setting', 'operator': '<=', 'value': 10.0}]}]}}
ACTUAL:
{'filters': {'operator': 'AND', 'conditions': [{'field': 'meta.category', 'operator': '==', 'value': 'Nanowires'}, {'operator': 'AND', 'conditions': [{'field': 'meta.scan_speed_setting', 'operator': '>=', 'value': 5.0}, {'field': 'meta.scan_speed_setting', 'operator': '<=', 'value': 10.0}]}, {'field': 'meta.working_distance_mm', 'operator': '>=', 'value': 100.0}]}}
====================================================================================================
DPIX-02 | dictionary_numeric
QUERY:
Show images with a spatial sampling of at least 50 nm per pixel.
EXPECTED:
{'filters': {'operator': 'AND', 'conditions': [{'field': 'meta.pixel_size_nm', 'operator': '>=', 'value': 50}]}}
ACTUAL:
{'filters': {'operator': 'AND', 'conditions': [{'field': 'meta.pixel_size_nm', 'operator': '<=', 'value': 50.0}]}}
```

### Qwen 3.8

```text
Accuracy: 99.49%
category
non_existing                    0.9
between                         1.0
categorical                     1.0
dictionary_categorical          1.0
dictionary_numeric              1.0
multiple_categorical            1.0
multiple_categorical_numeric    1.0
multiple_numeric                1.0
multiple_numeric_categorical    1.0
non_existing_multiple           1.0
numeric_approx                  1.0
numeric_between                 1.0
numeric_categorical             1.0
numeric_comparison              1.0
Name: correct, dtype: float64
Time: 2m 38.9s
```

*Errors:*
```text
====================================================================================================
NEM-09 | non_existing
QUERY:
Show images with a detector distance of approximately 10 mm.
EXPECTED:
{'filters': None}
ACTUAL:
{'filters': {'operator': 'AND', 'conditions': [{'operator': 'AND', 'conditions': [{'field': 'meta.working_distance_mm', 'operator': '>=', 'value': 9.0}, {'field': 'meta.working_distance_mm', 'operator': '<=', 'value': 11.0}]}]}}
```